# Dunnhumby 개발분할 5시드 — 결합 구성요소 재확인
결합모형에 넣을 CLV 구성요소 3개를 **같은 조건**에서 M1과 나란히 다시 학습합니다.
DAY 1~683 학습 → 684~690 개발평가, 현재 CLV 정의(q_N·q_V·q_C), 원 LightGCN BPR K=1(균일 음성), 100 epoch 고정, 시드 42~46.

| arm | 개입 지점 | 내용 |
|---|---|---|
| **M1** | — | 원 LightGCN |
| **M3-V 기여** | 그래프 | 엣지 가중 = q_V(u) × 해당 상품의 구매금액 비중 |
| **M2 N/V 개인이력 적합** | 표현 | q_N·q_V로 가중한 개인 구매이력 ↔ 후보상품 적합 블록 |
| **M4 보완 가중** | 손실 | 1 + 0.5·q_C·(1 − 가치성향·가격대 적합도) |

판정은 역할별로 사전 고정(5시드 중 4시드 이상 같은 방향 + 보호지표 98%)이며, 유의성 주장은 하지 않습니다.

**총 20회 학습(약 12~15시간)**. 끊기면 같은 노트북을 다시 실행하세요 — epoch 단위로 재개하고 완료된 arm은 캐시에서 재사용합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '9317b990de057cf4394db315bceb6bbd8d952c7c'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_component_recheck as recheck

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert recheck.CODE_VERSION == 'clv-component-recheck-dev5-v1'
cfg = recheck.configure_component_recheck()
summary = recheck.preflight_summary(cfg)
assert summary['seeds'] == [42, 43, 44, 45, 46]
assert summary['split'] == 'historical_development_days_684_690'
assert summary['loss']['negative_count'] == 1
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = recheck.run_component_recheck(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) 5시드 절대지표 평균과 95% 신뢰구간')
show(result_df.attrs['absolute_summary'])
print('2) M1 대비 동일 seed 차이의 5시드 평균')
show(result_df.attrs['paired_summary'])
print('3) 시드별 절대지표')
show(result_df)
print('4) 사전 판정')
print(json.dumps(result_df.attrs['reading'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
